In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


from src.data_creation import generate_energy_data
from src.data_viz import *
from src.models import *


c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
data = generate_energy_data()

In [3]:
run_eda(data)


── Test de Dickey-Fuller (ADF) ──
   Statistique : -1.6372
   p-value     : 0.4637
   Stationnaireé : NON — différenciation nécessaire


'output/01_eda.png'

In [5]:
def split_data(df: pd.DataFrame, test_days: int = 90):
    ts   = df.set_index("date")["consommation_kwh"]
    train = ts.iloc[:-test_days]
    test  = ts.iloc[-test_days:]
    print(f"\n── Split des données ──")
    print(f"   Train : {len(train)} jours  ({train.index[0].date()} → {train.index[-1].date()})")
    print(f"   Test  : {len(test)} jours   ({test.index[0].date()} → {test.index[-1].date()})")
    return train, test

In [6]:
def metrics(name: str, actual: pd.Series, pred: pd.Series) -> dict:
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f"   {name:<10} | MAE={mae:7.2f} | RMSE={rmse:7.2f} | MAPE={mape:5.2f}%")
    return {"Modèle": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)}
 

In [7]:
train, test = split_data(data, test_days=90)


── Split des données ──
   Train : 1005 jours  (2021-01-01 → 2023-10-02)
   Test  : 90 jours   (2023-10-03 → 2023-12-31)


In [8]:
pred_arima   = fit_arima(train, test)
pred_sarima  = fit_sarima(train, test)
pred_prophet = fit_prophet(train, test)
pred_lstm    = fit_lstm(train, test)
 
preds = {
        "ARIMA":   pred_arima,
        "SARIMA":  pred_sarima,
        "Prophet": pred_prophet,
        "LSTM":    pred_lstm,
    }


── ARIMA(2,1,2) ──


c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


   AIC : 8137.33

── SARIMA(2,1,2)(1,1,1,7) ──


c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
16:13:13 - cmdstanpy - INFO - Chain [1] start processing


   AIC : 7611.18

── Prophet ──


16:13:14 - cmdstanpy - INFO - Chain [1] done processing



── LSTM ──


c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
results = [metrics(name, test, pred) for name, pred in preds.items()]
results_df = pd.DataFrame(results)

   ARIMA      | MAE=  29.25 | RMSE=  36.07 | MAPE= 9.75%
   SARIMA     | MAE=  40.57 | RMSE=  49.09 | MAPE=13.43%
   Prophet    | MAE=   7.50 | RMSE=   9.62 | MAPE= 2.37%
   LSTM       | MAE=  31.11 | RMSE=  37.91 | MAPE=10.35%


In [10]:
results_df

,Modèle,MAE,RMSE,MAPE (%)
0,ARIMA,29.25,36.07,9.75
1,SARIMA,40.57,49.09,13.43
2,Prophet,7.50,9.62,2.37
3,LSTM,31.11,37.91,10.35


In [11]:
plot_forecasts(train, test, preds, "output/02_forecasts.png")
plot_metrics(results_df, "output/03_metrics.png")
plot_residuals(test, preds, "output/04_residuals.png")

c:\Users\antoi\Desktop\Vision\code\Deep-Learning\src\data_viz.py:180: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
c:\Users\antoi\Desktop\Vision\code\Deep-Learning\src\data_viz.py:181: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, dpi=150, bbox_inches="tight")


'output/04_residuals.png'